# EIA Country-Level Heavy Crude Oil Data Ingestion

## Purpose
This notebook ingests and processes country-level heavy crude oil production data from the U.S. Energy Information Administration (EIA).

## Workflow
1. **Data Ingestion**: Loads country-level heavy crude percentage data from Google Sheets into bronze layer (`workspace.bronze.eia_countrylvl_heavy_prod`)
2. **Data Cleaning**: Filters for "Percent Heavy by API" metric and selects years 2010-2024 (`workspace.silver.eia_countrylvl_heavy_prod_cleanedpercent`)
3. **Data Transformation**: Pivots data from wide format (year columns) to long format (year rows) using STACK (`workspace.silver.eia_countrylvl_heavy_prod_cleanedpivotpercent`)
4. **Calculate Heavy Oil Production**: Joins with total oil production data (`workspace.silver.eia_oilcond_production`) to calculate actual heavy oil production volumes (`workspace.silver.eia_countrylvl_heavy_prod_calculated`)
5. **Import Assumption Quality**: Loads data quality ratings for the heavy oil percentage assumptions (`workspace.silver.heavyprod_assumption_quality`)

## Output Tables
- `workspace.bronze.eia_countrylvl_heavy_prod` - Raw ingested data
- `workspace.silver.eia_countrylvl_heavy_prod_cleanedpercent` - Filtered and cleaned percentages
- `workspace.silver.eia_countrylvl_heavy_prod_cleanedpivotpercent` - Pivoted to long format
- `workspace.silver.eia_countrylvl_heavy_prod_calculated` - Heavy oil production volumes
- `workspace.silver.heavyprod_assumption_quality` - Data quality ratings

In [0]:
# import libraries
import pandas as pd
import sklearn
import numpy as np

Defining the google sheet url & tab id

In [0]:
# Original Google sheet URL (EIA PRODUCTION):
# https://docs.google.com/spreadsheets/d/1lf7Qd0MjLfp8C7h5HZuPlOXIvYdlDHUyUYsMUOPoufw/edit?gid=947317138#gid=947317138

# to CSV Export format: 
# https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}

sheet_id = "1lf7Qd0MjLfp8C7h5HZuPlOXIvYdlDHUyUYsMUOPoufw"
gid_p = "947317138" # sheet tab ID

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid_p}"

###Country-level Production - Ingestion


In [0]:

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid_p}"

# Load CSVs via pandas first (Serverless doesn't support direct HTTP reads with Spark)
df = spark.createDataFrame(pd.read_csv(url))

# Clean column names (Delta doesn't allow special characters)
def clean_column_names(df):
    for col in df.columns:
        clean_col = col.replace('(', '').replace(')', '').replace(',', '').replace(' ', '_').replace('/','per')
        if col != clean_col:
            df = df.withColumnRenamed(col, clean_col)
    return df

df = clean_column_names(df)

# Write as a Delta table in Unity Catalog (UC)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.eia_countrylvl_heavy_prod")

Check & view the imported table

In [0]:
%sql
SELECT * FROM workspace.bronze.eia_countrylvl_heavy_prod;

Country,Metric,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
United States,Production (kbd),null,null,null,null,null,null,null,null,9207.953296,8774.205438,8374.736962,8131.639273,8244.562,8707.440995,8551.534142,8596.590164,8571.580822,8648.372603,8687.60274,8878.950702,8971.558904,8680.10137,8348.972603,8139.817,7613.013699,7355.323288,7416.60274,7171.124525,6846.664918,6661.579071,6559.639504,6464.526363,6451.592236,6251.83397,5881.456584,5821.601095,5801.402738,5744.076712,5649.238356,5440.915301,5183.712329,5085.866,5073.898375,4999.669,5356.695871,5484.400063,5667.227447,6520.587402,7494.117715,8789.392,9446.493222,8851.519052,9371.356104,10964.08787,12248.01915,11307.56285,11311.29713,12004.21777,12943.38,13234.58562,13586.0869
United States,Percent Heavy by API,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.1450135371,0.1283192081,0.1135467729,0.1004749783,0.088908042,0.078672721,0.068435393,0.059909392,0.04609062,0.040900351,0.041003693,0.036239,0.031752762,0.032546463,0.032546463,null
United States,Growth rate %,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-0.13,-0.13,-0.13,-0.13,-0.13,-0.13,-0.13,-0.12,-0.23,-0.11,0.0,-0.12,-0.12,0.02,0.0,null
United States,Heavy Oil Production (kbd),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,795.3122519,727.2141381,740.3916571,752.9713145,781.4476331,743.1813257,605.757185,561.4322464,505.3416076,500.9482821,463.6518356,409.9100968,381.1670699,421.2612383,430.7389512,null
Russia,Production (kbd),null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,7656.756831,6753.153425,6382.923855,6166.004258,6008.000992,6089.760362,6021.242192,6052.362466,6476.82909,7010.512216,7676.054,8494.191011,9239.165847,9490.952775,9572.176647,9862.364362,9356.783607,9495.364932,9694.114466,9773.517808,9921.60929,10053.84384,10107.08767,10252.85479,10551.49727,10605.04932,10758.55483,10847.37415,9865.42172,10111.82793,10318.97148,10276.55934,9891.875412,9885.109
Russia,Percent Heavy by API,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.011922,0.013427,0.014674,0.016219,0.017583,0.019022,0.022655,0.023801,0.024986,0.02676,0.029536,null
Russia,Heavy Oil Production (kbd),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,120.5,137.6667,154.8333,172.0,189.1667,206.3333,223.5,240.6667,257.8333,275.0,292.1667,null
Saudi Arabia,Production (kbd),null,null,null,null,null,null,null,null,7596.137,8479.887671,7075.074,8576.967213,9245.019178,8300.712329,9531.956164,9900.15847,9814.948,6482.991781,5085.89589,4443.443,3387.821918,4870.016438,4264.991781,5085.95082,5064.156164,6410.046575,8115.153425,8216.202,7959.041096,7898.531507,7936.506849,7907.95082,8083.726027,8092.054795,7519.794521,7997.336066,7697.627397,7379.739726,9067.905479,9291.445355,9779.915,9642.118356,9190.088863,9261.25097,8250.112447,8417.012455,9620.287,9996.579962,9872.870454,9905.32092,

###Cleaning & Tranformation (PROD)

In [0]:
# To keep = col years 2010-2024, we need rows with "Metric" = "% Heavy by API"

# Load the production table from bronze schema
df = spark.table("workspace.bronze.eia_countrylvl_heavy_prod")

# Filter rows: keep only "Percent Heavy by API"
df_filtered = df.filter(df.Metric.contains("Percent Heavy by API"))

# Select columns: Country, Metric, and years 2010-2024
year_columns = [f"`{year}`" for year in range(2010, 2025)]
df_cleaned = df_filtered.select("Country", "Metric", *year_columns)

# Write as a Delta table in UC
df_cleaned.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.eia_countrylvl_heavy_prod_cleanedpercent")

print(f"Table created with {df_cleaned.count()} rows and {len(df_cleaned.columns)} columns")

Table created with 18 rows and 17 columns


In [0]:
%sql
SELECT * FROM workspace.silver.eia_countrylvl_heavy_prod_cleanedpercent;

Country,Metric,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
United States,Percent Heavy by API,0.1450135371,0.1283192081,0.1135467729,0.1004749783,0.088908042,0.078672721,0.068435393,0.059909392,0.04609062,0.040900351,0.041003693,0.036239,0.031752762,0.032546463,0.032546463
Russia,Percent Heavy by API,null,null,null,null,0.011922,0.013427,0.014674,0.016219,0.017583,0.019022,0.022655,0.023801,0.024986,0.02676,0.029536
Saudi Arabia,Percent Heavy by API,0.025,0.025,0.025,0.025,0.02,0.01,0.0,0.0,0.0,0.0,0.015,0.025,0.025,0.025,0.025
Canada,Percent Heavy by API,0.67,0.68,0.69,0.69,0.7,0.72,0.72,0.73,0.73,0.74,0.75,0.77,0.77,0.77,0.77
Iraq,Percent Heavy by API,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471,0.3471
China,Percent Heavy by API,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07
Iran,Percent Heavy by API,0.135013501,0.1327462718,0.1304790425,0.1304790425,0.1282118133,0.125944584,0.125944584,0.125944584,0.125944584,0.125944584,0.125944584,0.125944584,0.125944584,0.125944584,0.125944584
United Arab Emirates,Percent Heavy by API,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Brazil,Percent Heavy by API,0.35,0.35,0.35,0.35,0.35,0.35,0.35,0.35,0.35,0.35,0.35,0.35,0.35,0.35,0.35
Kuwait,Percent Heavy by API,null,null,null,null,null,null,0.01634695872,0.01656888667,0.01537334016,0.01457750721,0.07376138414,0.07092019185,0.06299573372,0.06384263821,0.06215452819


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.eia_countrylvl_heavy_prod_cleanedpivotpercent AS
SELECT 
  Country,
  Year,
  PercentHeavy
FROM (
  SELECT 
    Country,
    STACK(
      15,
      '2010', `2010`,
      '2011', `2011`,
      '2012', `2012`,
      '2013', `2013`,
      '2014', `2014`,
      '2015', `2015`,
      '2016', `2016`,
      '2017', `2017`,
      '2018', `2018`,
      '2019', `2019`,
      '2020', `2020`,
      '2021', `2021`,
      '2022', `2022`,
      '2023', `2023`,
      '2024', `2024`
    ) AS (Year, PercentHeavy)
  FROM workspace.silver.eia_countrylvl_heavy_prod_cleanedpercent
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM workspace.silver.eia_countrylvl_heavy_prod_cleanedpivotpercent;

Country,Year,PercentHeavy
United States,2010,0.1450135371
Russia,2010,null
Saudi Arabia,2010,0.025
Canada,2010,0.67
Iraq,2010,0.3471
China,2010,0.07
Iran,2010,0.135013501
United Arab Emirates,2010,0.0
Brazil,2010,0.35
Kuwait,2010,null


In [0]:
#verify eia data
spark.table("workspace.silver.eia_oilcond_production").display()

Country,Year,productName,unitName,value
Lithuania,2019,Crude oil including lease condensate,thousand barrels per day,0.7
Morocco,2019,Crude oil including lease condensate,thousand barrels per day,0.07
Mexico,2019,Crude oil including lease condensate,thousand barrels per day,1705.8468
Burma,2019,Crude oil including lease condensate,thousand barrels per day,8.766
Mongolia,2019,Crude oil including lease condensate,thousand barrels per day,18.462572
Malaysia,2019,Crude oil including lease condensate,thousand barrels per day,609.06506
Niger,2019,Crude oil including lease condensate,thousand barrels per day,15.0
Nigeria,2019,Crude oil including lease condensate,thousand barrels per day,1945.6432
Netherlands,2019,Crude oil including lease condensate,thousand barrels per day,18.157919
Norway,2019,Crude oil including lease condensate,thousand barrels per day,1437.1357


In [0]:
from pyspark.sql.functions import col, trim, upper

#prepare the eia data for joining
df_eia = spark.table("workspace.silver.eia_oilcond_production")

#only need columns country year and value from df_eia
df_eia = df_eia.select("Country", "Year", "value")

#prepare heavy oil % assumptions for joining 
df_heavy = spark.table("workspace.silver.eia_countrylvl_heavy_prod_cleanedpivotpercent")

#join the two dataframes on year and country
df_joined = df_eia.join(df_heavy, on=["Country", "Year"], how='right')

#create a new calculated column 'Heavy_Oil_Prod' by multiplying 'Oil_Produced' and 'PercentHeavy'
df_joined = df_joined.withColumn("Heavy_Oil_Prod", col("value") * col("PercentHeavy"))

#write to a new table
df_joined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.eia_countrylvl_heavy_prod_calculated")

In [0]:
#verify the table
spark.table("workspace.silver.eia_countrylvl_heavy_prod_calculated").display()

Country,Year,value,PercentHeavy,Heavy_Oil_Prod
United States,2010,5484.4,0.1450135371,795.3122287097618
Russia,2010,9694.114,null,null
Saudi Arabia,2010,8417.013,0.025,210.42531738281252
Canada,2010,2740.756,0.67,1836.3065893554688
Iraq,2010,2399.3025,0.3471,832.7978943603516
China,2010,4078.3604,0.07,285.48522460937505
Iran,2010,4080.419,0.135013501,550.9116473533682
United Arab Emirates,2010,2570.0,0.0,0.0
Brazil,2010,2054.668,0.35,719.1337890625
Kuwait,2010,2300.411,null,null


###Import assumptions quality & JOIN 

In [0]:
# https://docs.google.com/spreadsheets/d/1-pxqIIfcGQq9wFciYfc6rYeG8m8Ki5jQCwkscfWtz1U/edit?gid=0#gid=0
sheet_id = "1-pxqIIfcGQq9wFciYfc6rYeG8m8Ki5jQCwkscfWtz1U"
gid = "0"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

# Load CSVs via pandas first (Serverless doesn't support direct HTTP reads with Spark)
df_quality_pd = pd.read_csv(url, usecols=["Assumption Quality", "Country"])
df_quality = spark.createDataFrame(df_quality_pd)

# Clean column names (Delta doesn't allow special characters)
def clean_column_names(df):
    for col in df.columns:
        clean_col = col.replace('(', '').replace(')', '').replace(',', '').replace(' ', '_').replace('/','per')
        if col != clean_col:
            df = df.withColumnRenamed(col, clean_col)
    return df

df_quality = clean_column_names(df_quality)

# Write as a Delta table in UC
df_quality.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.heavyprod_assumption_quality")

Check imported assumption quality table

In [0]:
%sql
SELECT * FROM workspace.silver.heavyprod_assumption_quality;

Assumption_Quality,Country
FAIR,United States
POOR,Russia
FAIR,Saudi Arabia
GOOD,Canada
POOR,Iraq
POOR,China
POOR,Iran
GOOD,United Arab Emirates
POOR,Brazil
FAIR,Kuwait
